# 01 · 보유 데이터 현황

1단계는 **질환 라벨이 붙은 PPG**가 필요하다. 그 조건을 만족하는 것은 MIMIC-III-Ext-PPG 하나뿐이다.
2단계 사전학습에는 라벨이 필요 없으므로 보유한 PPG 전부를 쓴다.

이 노트북은 각 데이터셋의 존재 여부·표본화율·규모를 실측으로 확인한다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("01_dataset/01_data_inventory")
print("산출물 →", rep.dir)

## 1. 데이터 루트에 무엇이 있는가

In [ ]:
root = paths.data_root()
for p in sorted(root.iterdir()):
    if p.is_dir():
        print(f"{p.name}")

## 2. 단계별 사용 계획

표본화율은 EDF/WFDB 헤더에서 직접 읽은 값이다. 25 Hz는 박동당 24샘플이라 절흔 검출조차 되지 않는다.

In [ ]:
plan = pd.DataFrame([
    ("MIMIC-III-Ext-PPG",        125, "6,189명 · 5.3만 시간",      "ICD-10 1,092코드 · 리듬 16종", True,  True),
    ("MIMIC-III Waveform 원본",  125, "39,320 레코드 · 228만 시간", "",                            False, True),
    ("VitalDB",                  500, "6,388 케이스 · 126 GB",     "동맥압 동시기록",              False, True),
    ("MIMIC-IV Waveform",        125, "198명 · 8,969시간",         "",                            False, True),
    ("NFS",                      256, "398파일 · 3,607시간",       "연령·성별·BMI·수면설문",        False, True),
    ("PPGArrhythmiaDetection",   100, "46,827 세그먼트 · 130시간",  "부정맥 6클래스",               False, True),
    ("WildPPG",                  128, "부위 4종",                  "측정 부위",                    False, True),
    ("WF-PPG",                   100, "접촉압 6단계",              "접촉압",                       False, True),
    ("GalaxyPPG · Samsung",       25, "—",                        "—",                           False, False),
], columns=["dataset", "fs_Hz", "규모", "라벨", "1단계", "2단계"])
rep.table(plan, "dataset_plan.csv", "단계별 사용 계획")
rep.table(plan, "dataset_plan.csv", "단계별 사용 계획")
plan

## 3. Ext-PPG와 원본 waveform의 중복

Ext-PPG의 `record_id`는 MIMIC-III Waveform 원본과 같은 번호 체계다. 그래서 어느 원본 레코드에서
추출됐는지 정확히 대조된다. 겹치지 않는 레코드만 2단계 사전학습에 쓰면 평가 코호트로의 누출이 없다.

In [ ]:
src = paths.interim("ext_ppg_source_records.csv")
if src.exists():
    overlap = pd.read_csv(src)
    print("Ext-PPG가 추출된 원본 레코드 중 원본 덤프에 있는 것:", f"{len(overlap):,}개")
else:
    print("먼저 중복 확인을 수행해야 한다 —", src)

## 산출물

In [ ]:
rep.done("보유 PPG 현황과 단계별 사용 계획")
rep.summary()